# 04 · Delta-Hedge Backtest

Backtest a **short straddle delta-hedged** strategy using:
- **Live Deribit historical hourly price data** for the BTC index
- **Current surface parameters** (IV, strikes) from notebook 01
- **Realistic Deribit fee model** (0.03% options taker, 0.05% perp swap)

Strategy:
1. Sell one ATM straddle (short call + short put) at current market IV
2. Delta-hedge with the BTC perpetual swap every hour (or when |Δ| > 0.05)
3. Let the position expire and settle
4. Report full P&L attribution: Gamma / Theta / Fees

**Run notebook 01 first** to generate fresh market data.

**Outputs:** `results/backtest_pnl.png`, `results/backtest_summary.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from src.deribit_client import DeribitClient
from src.iv_calculator import compute_iv_surface
from src.surface_fit import VolSurface
from src.backtester import DeltaHedgeBacktester, FeeModel, BacktestResult

Path('../results').mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Libraries loaded.')

In [ ]:
# ── 1. Load the current option surface snapshot ──────────────────────────────
files = sorted(Path('../data/raw').glob('btc_surface_*.parquet'))
if not files:
    raise FileNotFoundError(
        'No BTC surface files found. Run notebook 01_data_collection.ipynb first.'
    )

snap = pd.read_parquet(files[-1])
print(f'Surface snapshot: {files[-1].name}')
print(f'  Spot:      ${snap["spot"].iloc[0]:,.2f}')
print(f'  Timestamp: {snap["timestamp"].iloc[0]}')
print(f'  Options:   {len(snap)} across {snap["expiry"].nunique()} expiries')

In [ ]:
# ── 2. Fetch historical hourly BTC prices from Deribit ───────────────────────
client = DeribitClient()

print('Fetching 90 days of hourly BTC prices from Deribit...')
price_history = client.get_index_price_history(
    index_name='btc_usd',
    resolution='60',       # 60-minute candles
    days_back=90,
)

print(f'  Retrieved: {len(price_history):,} hourly bars')
print(f'  From:  {price_history.index[0].strftime("%Y-%m-%d %H:%M UTC")}')
print(f'  To:    {price_history.index[-1].strftime("%Y-%m-%d %H:%M UTC")}')
print(f'  Start: ${price_history.iloc[0]:,.2f}')
print(f'  End:   ${price_history.iloc[-1]:,.2f}')
print(f'  Return: {100*(price_history.iloc[-1]/price_history.iloc[0]-1):.1f}%')
print(f'  Realised vol: {price_history.pct_change().std() * np.sqrt(365.25*24) * 100:.1f}% (hourly ann.)')

In [ ]:
# ── 3. Fit the surface to get ATM IV for each expiry ────────────────────────
surface = VolSurface(min_strikes=5)
surface.fit(snap, iv_col='calc_iv')

# Use the nearest expiry for the backtest straddle
sl     = surface.slices[0]
S0     = sl.F                     # current BTC forward (≈ spot)
T0     = sl.T                     # time to expiry in years
iv_atm = sl.atm_vol()             # ATM implied vol from calibrated SVI

print(f'Straddle parameters:')
print(f'  Expiry   : {sl.expiry_str}')
print(f'  T        : {T0:.4f} yr  ({T0*365.25:.1f} days)')
print(f'  Strike   : ${S0:,.2f}  (ATM)')
print(f'  ATM IV   : {iv_atm*100:.2f}%')
print(f'  RV (30d) : {price_history.pct_change().rolling(30*24).std().iloc[-1] * np.sqrt(365.25*24) * 100:.1f}%  (implied > realised → positive VRP expected)')

In [ ]:
# ── 4. Slice historical prices to match the option's remaining life ──────────
# Use the last T0 years of historical data as a realistic price path proxy.
# On a live desk, you would forward-test this as each day passes.
n_bars_needed  = max(int(T0 * 365.25 * 24), 48)   # at least 48 hours
backtest_prices = price_history.iloc[-n_bars_needed:].copy()

print(f'Backtest price path:')
print(f'  Bars: {len(backtest_prices):,} hourly')
print(f'  Start: ${backtest_prices.iloc[0]:,.2f}')
print(f'  End:   ${backtest_prices.iloc[-1]:,.2f}')
print(f'  Max drawdown from start: {100*(backtest_prices.min()/backtest_prices.iloc[0]-1):.1f}%')

In [ ]:
# ── 5. Run the delta-hedge backtest ─────────────────────────────────────────
bt = DeltaHedgeBacktester(
    fee_model=FeeModel(),          # Deribit 2024 fee schedule
    hedge_threshold=0.05,          # rebalance when |net Δ| > 0.05
    hedge_frequency_h=1.0,         # also rebalance every hour
    slippage_bps=1.0,              # 1bp slippage on perp hedge
)

bt.add_short_straddle(
    strike=S0,
    initial_T=T0,
    n_lots=1.0,
    spot=S0,
    iv=iv_atm,
    timestamp=backtest_prices.index[0],
    expiry_str=sl.expiry_str,
)

result = bt.run(backtest_prices, expiry_T_init=T0)

import json
print('=== BACKTEST RESULT ===')
print(json.dumps(result.summary(), indent=2, default=str))

In [ ]:
# ── 6. P&L Attribution ──────────────────────────────────────────────────────
attr = result.pnl_attribution()
print('\n=== P&L ATTRIBUTION ===')
print(f'  Gamma P&L  : ${attr.get("gamma_pnl", 0):>10,.2f}')
print(f'  Theta P&L  : ${attr.get("theta_pnl", 0):>10,.2f}')
print(f'  Delta P&L  : ${attr.get("delta_pnl", 0):>10,.2f}')
print(f'  Fee P&L    : ${attr.get("fee_pnl", 0):>10,.2f}')
print(f'  ─────────────────────────')
total = result.summary()['total_pnl_usd']
print(f'  TOTAL      : ${total:>10,.2f}')

pnl_df = result.pnl_df
print(f'\nRisk metrics:')
print(f'  Sharpe ratio   : {result.sharpe():.3f}')
print(f'  Max drawdown   : ${result.max_drawdown():,.2f}')
print(f'  Win rate       : {result.win_rate():.1%}')
print(f'  Steps run      : {len(pnl_df):,}')

In [ ]:
# ── 7. Full Backtest Dashboard ───────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 2, hspace=0.45, wspace=0.35)

# 1. BTC price path
ax1 = fig.add_subplot(gs[0, :])
backtest_prices.plot(ax=ax1, color='#2E86AB', lw=1.5, label='BTC/USD')
ax1.axhline(S0, color='red', ls='--', lw=1.2, label=f'Strike ${S0:,.0f}')
ax1.set_ylabel('BTC Price (USD)')
ax1.set_title('BTC Index Price — Backtest Period (Live Deribit Data)', fontweight='bold')
ax1.legend()

# 2. Cumulative P&L
ax2 = fig.add_subplot(gs[1, 0])
cum_pnl = result.cumulative_pnl()
cum_pnl.plot(ax=ax2, color='#2E86AB', lw=2)
ax2.axhline(0, color='black', lw=0.8)
ax2.fill_between(cum_pnl.index, cum_pnl, 0,
                 where=(cum_pnl >= 0), alpha=0.3, color='green')
ax2.fill_between(cum_pnl.index, cum_pnl, 0,
                 where=(cum_pnl < 0), alpha=0.3, color='red')
ax2.set_ylabel('Cumulative P&L ($)')
ax2.set_title('Cumulative P&L', fontweight='bold')

# 3. Step P&L bars
ax3 = fig.add_subplot(gs[1, 1])
step_pnl = pnl_df['total_pnl']
colors   = ['#2E86AB' if v >= 0 else '#E84855' for v in step_pnl]
ax3.bar(range(len(step_pnl)), step_pnl.values, color=colors, alpha=0.7, width=1.0)
ax3.axhline(0, color='black', lw=0.8)
ax3.set_xlabel('Hour')
ax3.set_ylabel('Hourly P&L ($)')
ax3.set_title('Hourly P&L', fontweight='bold')

# 4. P&L attribution bar chart
ax4 = fig.add_subplot(gs[2, 0])
components = {
    'Gamma': float(pnl_df['gamma_pnl'].sum()),
    'Theta': float(pnl_df['theta_pnl'].sum()),
    'Delta': float(pnl_df['delta_pnl'].sum()),
    'Fees':  float(pnl_df['fee_pnl'].sum()),
}
comp_colors = ['#E84855' if v < 0 else '#2E86AB' for v in components.values()]
ax4.barh(list(components.keys()), list(components.values()),
         color=comp_colors, alpha=0.85)
ax4.axvline(0, color='black', lw=0.8)
ax4.set_xlabel('P&L ($)')
ax4.set_title('P&L Attribution', fontweight='bold')
for i, (k, v) in enumerate(components.items()):
    ax4.text(v + (200 if v >= 0 else -200), i, f'${v:,.0f}',
             va='center', ha='left' if v >= 0 else 'right', fontsize=9)

# 5. Rolling portfolio IV (entry IV line)
ax5 = fig.add_subplot(gs[2, 1])
pnl_df['portfolio_iv'].mul(100).plot(ax=ax5, color='#F18F01', lw=1.5)
ax5.axhline(iv_atm * 100, color='red', ls='--', lw=1.2,
            label=f'Entry IV: {iv_atm*100:.1f}%')
ax5.set_ylabel('Portfolio IV (%)')
ax5.set_title('Portfolio IV Over Backtest', fontweight='bold')
ax5.legend()

spot_now = snap['spot'].iloc[0]
plt.suptitle(
    f'Short Straddle Delta-Hedge Backtest  |  BTC {sl.expiry_str}  |  '
    f'Strike ${S0:,.0f}  |  Entry IV {iv_atm*100:.1f}%  |  '
    f'Total P&L ${total:+,.0f}',
    fontsize=12, fontweight='bold', y=1.01
)
plt.savefig('../results/backtest_pnl.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved results/backtest_pnl.png')

In [ ]:
# ── 8. Save backtest summary ─────────────────────────────────────────────────
summary_row = pd.DataFrame([{
    'expiry':     sl.expiry_str,
    'strike':     S0,
    'entry_iv':   round(iv_atm * 100, 2),
    'T_days':     round(T0 * 365.25, 1),
    'n_bars':     len(backtest_prices),
    'total_pnl':  round(total, 2),
    'sharpe':     round(result.sharpe(), 4),
    'max_dd':     round(result.max_drawdown(), 2),
    'win_rate':   round(result.win_rate(), 4),
    'gamma_pnl':  round(float(pnl_df['gamma_pnl'].sum()), 2),
    'theta_pnl':  round(float(pnl_df['theta_pnl'].sum()), 2),
    'fee_pnl':    round(float(pnl_df['fee_pnl'].sum()), 2),
}])

out_path = '../results/backtest_summary.csv'
summary_row.to_csv(out_path, index=False)
print(f'Saved {out_path}')
display(summary_row)